In [2]:
import pandas as pd

In [3]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

def run_preprocessing(df: pd.DataFrame):
    """
    Final preaprocessing pipeline for shark tank data.
    Returns:
        X_scaled : Processed feature matrix (DataFrame)
        y_reg    : Regression target (Total Deal Amount)
        y_cls    : Classification target (Accepted Offer)
        y_shark  : Multi-label target (Individual shark investments)

    NaN handling summary:
        - shark_amt_cols      : fillna(0) — no investment means 0
        - shark_present_cols  : fillna(0) — not present means 0
        - financial_cols/ask_cols : to_numeric + fillna(median)
        - pitcher_cols (numeric): to_numeric + fillna(median)
        - Pitchers Average Age : categorical ('Young'/'Middle'/'Old'),
                                  label-encoded then NaN filled with mode
        - Cash Burn           : binary string ('yes'), converted to 0/1
                                  then NaN filled with 0
        - Started in          : numeric year, fillna(median)
        - Industry            : NaN rows get all-zero dummy columns (safe)
    """
    df = df.copy()

    shark_amt_cols = [
        'Namita Investment Amount', 'Vineeta Investment Amount',
        'Anupam Investment Amount', 'Aman Investment Amount',
        'Peyush Investment Amount', 'Ritesh Investment Amount',
        'Amit Investment Amount'
    ]
    shark_present_cols = [
        'Namita Present', 'Vineeta Present', 'Anupam Present',
        'Aman Present', 'Peyush Present', 'Ritesh Present',
        'Amit Present', 'Guest Present'
    ]

    # ── Shark amounts & presence ─────────────────────────────────────────────
    df[shark_amt_cols]     = df[shark_amt_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
    df[shark_present_cols] = df[shark_present_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
    df['sharks_present_count'] = df[shark_present_cols].sum(axis=1)

    y_shark = (df[shark_amt_cols] > 0).astype(int)
    y_shark.columns = [
        'Namita_Invested', 'Vineeta_Invested', 'Anupam_Invested',
        'Aman_Invested', 'Peyush_Invested', 'Ritesh_Invested', 'Amit_Invested'
    ]

    # ── Derived targets ──────────────────────────────────────────────────────
    df['Total Deal Amount']       = df[shark_amt_cols].sum(axis=1)
    df['Number of Sharks in Deal'] = (df[shark_amt_cols] > 0).sum(axis=1)

    # ── Financial & ask columns (numeric) ───────────────────────────────────
    financial_cols = ['Yearly Revenue', 'Monthly Sales', 'Gross Margin',
                      'Net Margin', 'EBITDA', 'SKUs']
    ask_cols       = ['Original Ask Amount', 'Original Offered Equity', 'Valuation Requested']

    num_cols = financial_cols + ask_cols
    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())

    # ── Cash Burn: binary string column ('yes' / NaN) ───────────────────────
    # Convert to 1/0; NaN (no data) treated as 0 (not burning cash / unknown)
    df['Cash Burn'] = df['Cash Burn'].apply(
        lambda x: 1 if str(x).strip().lower() == 'yes' else 0
    )

    # ── Accepted Offer ───────────────────────────────────────────────────────
    df['Accepted Offer']          = df['Accepted Offer'].fillna(0).astype(int)
    df['Original Offered Equity'] = df['Original Offered Equity'].replace(0, 1e-6)
    df['Original Ask Amount']     = df['Original Ask Amount'].replace(0, 1e-6)

    # ── Engineered financial ratios ──────────────────────────────────────────
    df['ask_per_equity']      = df['Original Ask Amount'] / df['Original Offered Equity']
    df['valuation_ask_ratio'] = df['Valuation Requested']  / df['Original Ask Amount']
    df['revenue_ask_ratio']   = df['Yearly Revenue']       / df['Original Ask Amount']
    df['is_revenue_positive'] = (df['Yearly Revenue'] > 0).astype(int)

    y_reg = df['Total Deal Amount']
    y_cls = df['Accepted Offer']

    # ── Context columns ──────────────────────────────────────────────────────
    context_cols = ['Season Number', 'Season Start', 'Season End', 'Started in', 'Industry']

    # 'Started in': numeric year — fill missing with median year
    df['Started in'] = pd.to_numeric(df['Started in'], errors='coerce')
    df['Started in'] = df['Started in'].fillna(df['Started in'].median())

    # ── Pitcher columns ──────────────────────────────────────────────────────
    numeric_pitcher_cols = [
        'Number of Presenters', 'Male Presenters', 'Female Presenters',
        'Transgender Presenters', 'Couple Presenters'
    ]
    gender_cols = ['Male Presenters', 'Female Presenters',
                   'Transgender Presenters', 'Couple Presenters']

    df[numeric_pitcher_cols] = df[numeric_pitcher_cols].apply(pd.to_numeric, errors='coerce')
    df[numeric_pitcher_cols] = df[numeric_pitcher_cols].fillna(df[numeric_pitcher_cols].median())

    # 'Pitchers Average Age': categorical string ('Young' / 'Middle' / 'Old')
    # Label-encode: Young=0, Middle=1, Old=2; NaN → mode
    age_map = {'Young': 0, 'Middle': 1, 'Old': 2}
    df['Pitchers Average Age'] = df['Pitchers Average Age'].map(age_map)
    age_mode = df['Pitchers Average Age'].mode(dropna=True)
    df['Pitchers Average Age'] = df['Pitchers Average Age'].fillna(
        age_mode[0] if len(age_mode) > 0 else 1  # default to 'Middle' if no mode
    )

    pitcher_cols = numeric_pitcher_cols + ['Pitchers Average Age']

    # ── Engineered pitcher features ──────────────────────────────────────────
    df['team_gender_diversity'] = (
        (df[gender_cols] > 0).sum(axis=1) / df['Number of Presenters'].replace(0, 1)
    )
    df['season_number_norm'] = (
        (df['Season Number'] - df['Season Number'].min()) /
        (df['Season Number'].max() - df['Season Number'].min())
    )

    # ── Assemble feature matrix ──────────────────────────────────────────────
    X = pd.concat([
        df[context_cols + pitcher_cols + ['team_gender_diversity', 'season_number_norm']],
        df[financial_cols + ['Cash Burn'] + ask_cols + [
            'ask_per_equity', 'valuation_ask_ratio', 'revenue_ask_ratio',
            'is_revenue_positive', 'Number of Sharks in Deal'
        ]],
        df[shark_amt_cols + shark_present_cols + ['sharks_present_count']]
    ], axis=1)

    # Industry: get_dummies handles NaN rows as all-zero (safe)
    X = pd.get_dummies(X, columns=['Industry'], drop_first=True)
    X = X.drop(['Season Start', 'Season End'], axis=1, errors='ignore')

    # Final NaN check — should be zero
    remaining_nans = X.isnull().sum().sum()
    if remaining_nans > 0:
        print(f"⚠️  Warning: {remaining_nans} NaN values remain in X. Filling with column medians.")
        X = X.fillna(X.median(numeric_only=True))

    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

    return X_scaled, y_reg, y_cls, y_shark


In [4]:
# ── Update this path to your dataset ────────────────────────────────────────
DATA_PATH = '/home/varnan/Works/Projects/Shark-tank-India-Decision-Modeling/Shark Tank India.csv'   # <-- change this

raw_df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {raw_df.shape}')
raw_df.head(3)

Dataset shape: (789, 80)


,Season Number,Startup Name,Episode Number,Pitch Number,Season Start,Season End,Original Air Date,Episode Title,Anchor,Industry,...,Invested Guest Name,All Guest Names,Namita Present,Vineeta Present,Anupam Present,Aman Present,Peyush Present,Ritesh Present,Amit Present,Guest Present
0,1,BluePineFoods,1,1,20-Dec-21,4-Feb-22,20-Dec-21,Badlegi Business Ki Tasveer,Rannvijay Singh,Food and Beverage,...,Ashneer Grover,Ashneer Grover,1.0,1.0,1.0,1.0,NaN,NaN,NaN,1.0
1,1,BoozScooters,1,2,20-Dec-21,4-Feb-22,20-Dec-21,Badlegi Business Ki Tasveer,Rannvijay Singh,Vehicles/Electrical Vehicles,...,Ashneer Grover,Ashneer Grover,1.0,1.0,1.0,1.0,NaN,NaN,NaN,1.0
2,1,HeartUpMySleeves,1,3,20-Dec-21,4-Feb-22,20-Dec-21,Badlegi Business Ki Tasveer,Rannvijay Singh,Beauty/Fashion,...,NaN,Ashneer Grover,1.0,1.0,1.0,1.0,NaN,NaN,NaN,1.0


In [5]:
X_scaled, y_reg, y_cls, y_shark = run_preprocessing(raw_df)

print(f'X shape      : {X_scaled.shape}')
print(f'y_cls dist   : {y_cls.value_counts().to_dict()}  (0=No Deal, 1=Deal)')
print(f'y_reg range  : {y_reg.min():.0f} – {y_reg.max():.0f}')
print(f'y_shark shape: {y_shark.shape}')
y_shark.sum().rename('investments_count')

X shape      : (789, 58)
y_cls dist   : {1: 434, 0: 355}  (0=No Deal, 1=Deal)
y_reg range  : 0 – 500
y_shark shape: (789, 7)


Namita_Invested     135
Vineeta_Invested     95
Anupam_Invested     117
Aman_Invested       162
Peyush_Invested     103
Ritesh_Invested      59
Amit_Invested        38
Name: investments_count, dtype: int64

In [6]:
X_scaled.describe()

,Season Number,Started in,Number of Presenters,Male Presenters,Female Presenters,Transgender Presenters,Couple Presenters,Pitchers Average Age,team_gender_diversity,season_number_norm,...,Industry_Food and Beverage,Industry_Green/CleanTech,Industry_Hardware,Industry_Lifestyle/Home,Industry_Liquor/Alcohol,Industry_Manufacturing,Industry_Medical/Health,Industry_Others,Industry_Technology/Software,Industry_Vehicles/Electrical Vehicles
count,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,789.000000,789.0,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,...,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,789.000000,7.890000e+02,7.890000e+02
mean,7.204489e-17,9.131690e-15,-1.080673e-16,-1.260786e-16,0.000000,0.0,9.005611e-18,-4.502806e-17,7.204489e-17,7.204489e-17,...,-1.801122e-17,-4.052525e-17,-1.350842e-17,-1.801122e-17,1.125701e-17,-6.303928e-17,3.377104e-17,0.000000,4.502806e-17,-4.953086e-17
std,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634,0.0,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,...,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634,1.000634e+00,1.000634e+00
min,-1.419256e+00,-8.535374e+00,-1.246609e+00,-1.783372e+00,-1.627713,0.0,-4.461929e-01,-1.868357e+00,-1.851783e+00,-1.419256e+00,...,-5.220926e-01,-1.483938e-01,-5.041127e-02,-2.656244e-01,-8.753762e-02,-2.656244e-01,-3.046273e-01,-0.087538,-3.428757e-01,-1.771230e-01
25%,-7.064659e-01,-2.489940e-01,-1.246609e+00,-6.058986e-01,0.209577,0.0,-4.461929e-01,4.827333e-01,-9.861710e-01,-7.064659e-01,...,-5.220926e-01,-1.483938e-01,-5.041127e-02,-2.656244e-01,-8.753762e-02,-2.656244e-01,-3.046273e-01,-0.087538,-3.428757e-01,-1.771230e-01
50%,6.323864e-03,9.627184e-02,-1.577986e-03,-6.058986e-01,0.209577,0.0,-4.461929e-01,4.827333e-01,-2.648278e-01,6.323864e-03,...,-5.220926e-01,-1.483938e-01,-5.041127e-02,-2.656244e-01,-8.753762e-02,-2.656244e-01,-3.046273e-01,-0.087538,-3.428757e-01,-1.771230e-01
75%,7.191136e-01,4.415377e-01,-1.577986e-03,5.715743e-01,0.209577,0.0,-4.461929e-01,4.827333e-01,4.565155e-01,7.191136e-01,...,-5.220926e-01,-1.483938e-01,-5.041127e-02,-2.656244e-01,-8.753762e-02,-2.656244e-01,-3.046273e-01,-0.087538,-3.428757e-01,-1.771230e-01
max,1.431903e+00,1.822601e+00,4.978546e+00,5.281466e+00,5.721447,0.0,2.241183e+00,2.833823e+00,1.899202e+00,1.431903e+00,...,1.915369e+00,6.738825e+00,1.983683e+01,3.764715e+00,1.142366e+01,3.764715e+00,3.282700e+00,11.423660,2.916509e+00,5.645795e+00


In [7]:
pd.set_option('display.max_columns', None)

In [8]:
X_scaled.describe()

,Season Number,Started in,Number of Presenters,Male Presenters,Female Presenters,Transgender Presenters,Couple Presenters,Pitchers Average Age,team_gender_diversity,season_number_norm,Yearly Revenue,Monthly Sales,Gross Margin,Net Margin,EBITDA,SKUs,Cash Burn,Original Ask Amount,Original Offered Equity,Valuation Requested,ask_per_equity,valuation_ask_ratio,revenue_ask_ratio,is_revenue_positive,Number of Sharks in Deal,Namita Investment Amount,Vineeta Investment Amount,Anupam Investment Amount,Aman Investment Amount,Peyush Investment Amount,Ritesh Investment Amount,Amit Investment Amount,Namita Present,Vineeta Present,Anupam Present,Aman Present,Peyush Present,Ritesh Present,Amit Present,Guest Present,sharks_present_count,Industry_Animal/Pets,Industry_Beauty/Fashion,Industry_Business Services,Industry_Children/Education,Industry_Electronics,Industry_Entertainment,Industry_Fitness/Sports/Outdoors,Industry_Food and Beverage,Industry_Green/CleanTech,Industry_Hardware,Industry_Lifestyle/Home,Industry_Liquor/Alcohol,Industry_Manufacturing,Industry_Medical/Health,Industry_Others,Industry_Technology/Software,Industry_Vehicles/Electrical Vehicles
count,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,789.000000,789.0,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,789.000000,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,7.890000e+02,789.000000,7.890000e+02,7.890000e+02
mean,7.204489e-17,9.131690e-15,-1.080673e-16,-1.260786e-16,0.000000,0.0,9.005611e-18,-4.502806e-17,7.204489e-17,7.204489e-17,-2.251403e-17,-1.575982e-17,-7.204489e-17,2.476543e-16,-5.909932e-17,2.251403e-18,-5.403367e-17,-1.575982e-17,1.170729e-16,-1.350842e-17,-3.377104e-18,-4.052525e-17,-1.125701e-17,-4.727946e-17,9.005611e-17,-9.005611e-18,-2.701683e-17,9.005611e-18,-2.701683e-17,1.801122e-17,1.801122e-17,-1.801122e-17,-9.005611e-18,9.906172e-17,-1.080673e-16,-9.906172e-17,-7.204489e-17,3.602245e-17,3.602245e-17,0.000000,-5.403367e-17,-3.827385e-17,-7.767340e-17,-4.502806e-17,-5.403367e-17,-2.251403e-17,4.052525e-17,-2.251403e-17,-1.801122e-17,-4.052525e-17,-1.350842e-17,-1.801122e-17,1.125701e-17,-6.303928e-17,3.377104e-17,0.000000,4.502806e-17,-4.953086e-17
std,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634,0.0,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634,1.000634e+00,1.000634e+00
min,-1.419256e+00,-8.535374e+00,-1.246609e+00,-1.783372e+00,-1.627713,0.0,-4.461929e-01,-1.868357e+00,-1.851783e+00,-1.419256e+00,-3.932704e-01,-3.217500e-01,-5.413669e+00,-3.818853e+00,-8.913826e+00,-2.376531e-01,-4.193530e-01,-1.160084e-01,-9.712100e-01,-6.902764e-01,-6.902763e-01,-1.136484e+00,-3.630957e-02,-5.029911e+00,-7.998386e-01,-3.634660e-01,-3.068942e-01,-3.281478e-01,-3.534288e-01,-2.189644e-01,-2.401934e-01,-1.790661e-01,-1.763342e+00,-1.236465e+00,-2.081125e+00,-2.191226e+00,-9.836570e-01,-5.181590e-01,-4.864947e-01,-0.885830,-3.309774e+00,-1.133004e-01,-5.003960e-01,

NameError: name 'shark_amt_cols' is not defined